In [5]:
import pymysql
from pymysql.err import OperationalError

# ==========================================
# 请修改以下配置为你自己的 MySQL 连接信息
# ==========================================
config = {
    "host": "localhost",            # MySQL 服务器地址
    "port": 3306,                   # 端口号（默认 3306）
    "user": "root",                 # 用户名  ← 修改这里
    "password": "8888",    # 密码    ← 修改这里
    "database": "db_0623_1",          # 数据库名 ← 修改这里
    "charset": "utf8mb4",           # 字符编码
    "cursorclass": pymysql.cursors.DictCursor  # 以字典形式返回结果
}

# 建立连接
try:
    connection = pymysql.connect(**config)
    print("✅ 成功连接到 MySQL 数据库！")
except OperationalError as e:
    code, msg = e.args
    print(f"❌ 连接失败 (错误码 {code}): {msg}")
    print("请检查 config 中的连接信息是否正确")
    raise

# 使用 with 语句自动管理游标
with connection.cursor() as cursor:
    # 查询 MySQL 版本
    cursor.execute("SELECT VERSION() AS version")
    result = cursor.fetchone()
    print(f"📌 MySQL 版本: {result['version']}")

    # 查看所有数据库
    cursor.execute("SHOW DATABASES")
    databases = cursor.fetchall()
    print(f"\n📂 共有 {len(databases)} 个数据库:")
    for db in databases:
        print(f"   • {db['Database']}")

# 关闭连接
connection.close()
print("\n✅ 连接已关闭")

✅ 成功连接到 MySQL 数据库！
📌 MySQL 版本: 8.4.6

📂 共有 7 个数据库:
   • crashcourse
   • data
   • db_0623_1
   • information_schema
   • mysql
   • performance_schema
   • sys

✅ 连接已关闭


## 封装成类的写法（PyMySQL 版本）

讲义上的代码混用了 `mysql.connector` 和 `pymysql`，以下是修正后的 PyMySQL 版本：

In [4]:
import pymysql
from pymysql.err import OperationalError


class MySQLCRUD:
    """MySQL 数据库增删改查封装类（PyMySQL 版）"""

    def __init__(self, host, database, user, password, port=3306):
        self.host = host
        self.database = database
        self.user = user
        self.password = password
        self.port = port
        self.connection = None
        self.cursor = None

    def connect(self):
        """建立数据库连接"""
        try:
            self.connection = pymysql.connect(
                host=self.host,
                database=self.database,
                user=self.user,
                password=self.password,
                port=self.port,
                charset="utf8mb4",
                cursorclass=pymysql.cursors.DictCursor
            )
            self.cursor = self.connection.cursor()
            print(f"✅ 成功连接到 MySQL 数据库: {self.database}")
            return True
        except OperationalError as e:
            code, msg = e.args
            print(f"❌ 连接失败 (错误码 {code}): {msg}")
            return False

    def disconnect(self):
        """断开数据库连接"""
        if self.connection:
            if self.cursor:
                self.cursor.close()
            self.connection.close()
            print("🔒 数据库连接已关闭")

    def query(self, sql, params=None):
        """查询数据，返回字典列表"""
        self.cursor.execute(sql, params or ())
        return self.cursor.fetchall()

    def execute(self, sql, params=None):
        """执行单条插入/更新/删除，返回受影响行数"""
        affected = self.cursor.execute(sql, params or ())
        self.connection.commit()
        return affected

    def insert_many(self, sql, params_list):
        """批量插入多条数据
        sql: INSERT 语句
        params_list: 参数列表，如 [(v1,v2), (v3,v4), ...]
        """
        affected = self.cursor.executemany(sql, params_list)
        self.connection.commit()
        return affected


# ====== 使用示例 ======
if __name__ == "__main__":
    db = MySQLCRUD(
        host="localhost",
        database="db_0623_1",
        user="root",
        password="8888"
    )

    if db.connect():
        # 查看所有表
        tables = db.query("SHOW TABLES")
        print(f"📋 数据库中共有 {len(tables)} 张表:")
        for t in tables:
            # 兼容不同 MySQL 版本中 SHOW TABLES 的字段名
            table_name = list(t.values())[0]
            print(f"   📄 {table_name}")

        # 断开连接
        db.disconnect()

✅ 成功连接到 MySQL 数据库: db_0623_1
📋 数据库中共有 5 张表:
   📄 course_sys
   📄 score
   📄 student
   📄 student_sys
   📄 tab_stu1
🔒 数据库连接已关闭


## 增删改查（CRUD）操作示例

In [9]:
# 先查看 student 表的结构
db.connect()
desc = db.query("DESC student")
print("📋 student 表结构:")
print(f"{'字段名':<15} {'类型':<15} {'允许空':<8} {'键':<8} {'默认值':<10}")
print("-" * 60)
for col in desc:
    print(f"{col['Field']:<15} {col['Type']:<15} {col['Null']:<8} {col['Key']:<8} {str(col['Default']):<10}")
print()

✅ 成功连接到 MySQL 数据库: db_0623_1
📋 student 表结构:
字段名             类型              允许空      键        默认值       
------------------------------------------------------------
Id              int             NO       PRI      None      
Name            varchar(20)     NO                None      
Sex             varchar(4)      YES               None      
Birth           year            YES               None      
Department      varchar(20)     NO                None      
Address         varchar(50)     YES               None      



In [10]:
# ===== 增 (Create) =====
print("=" * 50)
print("📝 插入新学生记录")
db.execute(
    "INSERT INTO student (Id, Name, Sex, Birth, Department, Address) VALUES (%s, %s, %s, %s, %s, %s)",
    (101, "张三", "男", 2002, "计算机系", "北京市海淀区")
)
print("✅ 插入成功\n")

# ===== 查 (Read) =====
print("📖 查询所有学生:")
students = db.query("SELECT * FROM student")
for s in students:
    print(f"   {s['Id']}  {s['Name']}  {s['Sex']}  {s['Birth']}  {s['Department']}  {s['Address']}")
print()

# ===== 改 (Update) =====
print("✏️ 更新学生信息（将张三的院系改为'信息工程系'）")
db.execute(
    "UPDATE student SET Department = %s WHERE Id = %s",
    ("信息工程系", 101)
)
# 查看更新结果
updated = db.query("SELECT Id, Name, Department FROM student WHERE Id = %s", (101,))
print(f"✅ 更新结果: {updated}\n")

# ===== 删 (Delete) =====
print("🗑️ 删除刚才插入的测试数据")
db.execute("DELETE FROM student WHERE Id = %s", (101,))
remaining = db.query("SELECT COUNT(*) AS cnt FROM student")
print(f"✅ 删除成功，当前 student 表共 {remaining[0]['cnt']} 条记录\n")

# 断开连接
db.disconnect()

📝 插入新学生记录
✅ 插入成功

📖 查询所有学生:
   101  张三  男  2002  计算机系  北京市海淀区
   901  张老大  男  2005  计算机系  北京市海淀区
   902  张老二  男  2006  中文系  北京市昌平区
   903  张三  女  2002  中文系  湖南省永州市
   904  李四  男  2003  英语系  辽宁省阜新市
   905  王五  女  2004  英语系  福建省厦门市
   906  王六  男  2002  计算机系  湖南省衡阳市

✏️ 更新学生信息（将张三的院系改为'信息工程系'）
✅ 更新结果: [{'Id': 101, 'Name': '张三', 'Department': '信息工程系'}]

🗑️ 删除刚才插入的测试数据
✅ 删除成功，当前 student 表共 6 条记录

🔒 数据库连接已关闭


In [20]:
# 查看 course_sys 表的结构和数据
db.connect()

print("📋 course_sys 表结构:")
desc = db.query("DESC course_sys")
for col in desc:
    print(f"   {col['Field']:12}  {col['Type']:15}  {col['Key']}")

print("\n📖 course_sys 当前数据:")
courses = db.query("SELECT * FROM course_sys")
for c in courses:
    print(f"   {list(c.values())}")

✅ 成功连接到 MySQL 数据库: db_0623_1
📋 course_sys 表结构:
   cNo           varchar(10)      PRI
   cName         varchar(10)      
   createUser    varchar(50)      
   createTime    datetime         
   updateUser    varchar(50)      
   updateTime    datetime         

📖 course_sys 当前数据:
   ['C-101', '人工智能', 'admin', datetime.datetime(2026, 6, 23, 16, 41, 50), None, None]
   ['C-102', 'Python', 'teacher', datetime.datetime(2026, 6, 23, 16, 41, 50), 'admin', datetime.datetime(2026, 6, 23, 16, 41, 50)]
   ['C-103', '数据库', 'admin', datetime.datetime(2026, 6, 23, 16, 41, 50), None, None]
   ['C-104', 'PHP', 'teacher', datetime.datetime(2026, 6, 23, 16, 41, 50), None, None]
   ['C-105', 'HTML5', 'admin', datetime.datetime(2026, 6, 23, 16, 41, 50), 'teacher', datetime.datetime(2026, 6, 23, 16, 41, 50)]
   ['C-106', '高等数学', 'admin', datetime.datetime(2026, 6, 24, 15, 0, 28), None, None]


In [22]:
# 插入新课程 C-107 信号与系统（C-106 已存在）
sql = "INSERT INTO course_sys (cNo, cName, createUser, createTime) VALUES (%s, %s, %s, NOW())"
db.execute(sql, ("C-107", "信号与系统", "admin"))
print("✅ 插入成功")

# 查看插入后的数据
courses = db.query("SELECT * FROM course_sys")
print("\n📖 course_sys 更新后的数据:")
for c in courses:
    print(f"   {list(c.values())}")

db.disconnect()

✅ 插入成功

📖 course_sys 更新后的数据:
   ['C-101', '人工智能', 'admin', datetime.datetime(2026, 6, 23, 16, 41, 50), None, None]
   ['C-102', 'Python', 'teacher', datetime.datetime(2026, 6, 23, 16, 41, 50), 'admin', datetime.datetime(2026, 6, 23, 16, 41, 50)]
   ['C-103', '数据库', 'admin', datetime.datetime(2026, 6, 23, 16, 41, 50), None, None]
   ['C-104', 'PHP', 'teacher', datetime.datetime(2026, 6, 23, 16, 41, 50), None, None]
   ['C-105', 'HTML5', 'admin', datetime.datetime(2026, 6, 23, 16, 41, 50), 'teacher', datetime.datetime(2026, 6, 23, 16, 41, 50)]
   ['C-106', '高等数学', 'admin', datetime.datetime(2026, 6, 24, 15, 0, 28), None, None]
   ['C-107', '信号与系统', 'admin', datetime.datetime(2026, 6, 24, 15, 7, 37), None, None]
🔒 数据库连接已关闭


## 批量插入数据（insert_many）

In [24]:
# 批量插入多条课程
db.connect()

sql = "INSERT INTO course_sys (cNo, cName, createUser, createTime) VALUES (%s, %s, %s, NOW())"
new_courses = [
    ("C-201", "数据结构", "admin"),
    ("C-202", "操作系统", "admin"),
    ("C-203", "计算机网络", "admin"),
]

affected = db.insert_many(sql, new_courses)
print(f"✅ 成功插入 {affected} 条记录\n")

# 查看最终数据
courses = db.query("SELECT * FROM course_sys")
print("📖 course_sys 全部数据:")
for c in courses:
    print(f"   {list(c.values())}")

db.disconnect()

✅ 成功连接到 MySQL 数据库: db_0623_1
✅ 成功插入 3 条记录

📖 course_sys 全部数据:
   ['C-101', '人工智能', 'admin', datetime.datetime(2026, 6, 23, 16, 41, 50), None, None]
   ['C-102', 'Python', 'teacher', datetime.datetime(2026, 6, 23, 16, 41, 50), 'admin', datetime.datetime(2026, 6, 23, 16, 41, 50)]
   ['C-103', '数据库', 'admin', datetime.datetime(2026, 6, 23, 16, 41, 50), None, None]
   ['C-104', 'PHP', 'teacher', datetime.datetime(2026, 6, 23, 16, 41, 50), None, None]
   ['C-105', 'HTML5', 'admin', datetime.datetime(2026, 6, 23, 16, 41, 50), 'teacher', datetime.datetime(2026, 6, 23, 16, 41, 50)]
   ['C-106', '高等数学', 'admin', datetime.datetime(2026, 6, 24, 15, 0, 28), None, None]
   ['C-107', '信号与系统', 'admin', datetime.datetime(2026, 6, 24, 15, 7, 37), None, None]
   ['C-201', '数据结构', 'admin', datetime.datetime(2026, 6, 24, 15, 9, 43), None, None]
   ['C-202', '操作系统', 'admin', datetime.datetime(2026, 6, 24, 15, 9, 43), None, None]
   ['C-203', '计算机网络', 'admin', datetime.datetime(2026, 6, 24, 15, 9, 43), None

In [11]:
from mysql_db import MySqlHelper


class Student:
    def __init__(self, mysql_helper):
        self.helper = mysql_helper
        self.table = "student"  # 对应 db_0623_1 中的 student 表

    def student_menu(self):
        while True:
            print("-----------学生信息菜单----------")
            print("\n 1) 查询所有学生信息")
            print(" 2) 新增学生信息")
            print(" 3) 修改学生信息")
            print(" 4) 删除学生信息")
            print(" 5) 返回上一层菜单")
            no = input("请输入选择的序号：")

            if no == '1':
                self.show_students()
                input("按任意键返回上层菜单")
            elif no == '2':
                self.add_student()
                input("按任意键返回上层菜单")
            elif no == '3':
                self.edit_student()
                input("按任意键返回上层菜单")
            elif no == '4':
                self.delete_student()
                input("按任意键返回上层菜单")
            elif no == '5':
                break
            else:
                print("输入错误！请重新输入！")

    def show_students(self):
        """查询所有学生"""
        sql = "SELECT * FROM student"
        res = self.helper.find_all(sql)

        if not res:
            print("暂无学生数据")
            return

        print(f"\n{'='*65}")
        print(f"{'ID':<6} {'姓名':<10} {'性别':<6} {'出生':<8} {'院系':<16} {'地址':<20}")
        print(f"{'-'*65}")
        for stu in res:
            print(f"{stu['Id']:<6} {stu['Name']:<10} {stu['Sex']:<6} {stu['Birth']:<8} {stu['Department']:<16} {stu['Address']:<20}")
        print(f"{'='*65}")
        print(f"共 {len(res)} 条记录\n")

    def add_student(self):
        """新增学生"""
        print("\n--- 新增学生 ---")
        sid = int(input("学号(ID): "))
        name = input("姓名: ")
        sex = input("性别(男/女): ")
        birth = int(input("出生年份: "))
        dept = input("院系: ")
        addr = input("地址: ")

        sql = "INSERT INTO student (Id, Name, Sex, Birth, Department, Address) VALUES (%s, %s, %s, %s, %s, %s)"
        try:
            self.helper.execute(sql, (sid, name, sex, birth, dept, addr))
            print(f"✅ 新增成功: {sid} {name}")
        except Exception as e:
            print(f"❌ 新增失败: {e}")

    def edit_student(self):
        """修改学生"""
        print("\n--- 修改学生 ---")
        sid = int(input("请输入要修改的学号(ID): "))

        sql = "SELECT * FROM student WHERE Id = %s"
        stu = self.helper.find_one(sql, (sid,))
        if not stu:
            print(f"❌ 未找到学号 {sid}")
            return

        print(f"原信息: {stu}")
        name = input(f"姓名({stu['Name']}): ") or stu['Name']
        sex = input(f"性别({stu['Sex']}): ") or stu['Sex']
        birth_str = input(f"出生年份({stu['Birth']}): ")
        birth = int(birth_str) if birth_str else stu['Birth']
        dept = input(f"院系({stu['Department']}): ") or stu['Department']
        addr = input(f"地址({stu['Address']}): ") or stu['Address']

        sql = "UPDATE student SET Name=%s, Sex=%s, Birth=%s, Department=%s, Address=%s WHERE Id=%s"
        self.helper.execute(sql, (name, sex, birth, dept, addr, sid))
        print(f"✅ 修改成功: 学号 {sid}")

    def delete_student(self):
        """删除学生"""
        print("\n--- 删除学生 ---")
        sid = int(input("请输入要删除的学号(ID): "))

        sql = "SELECT * FROM student WHERE Id = %s"
        stu = self.helper.find_one(sql, (sid,))
        if not stu:
            print(f"❌ 未找到学号 {sid}")
            return

        print(f"即将删除: {stu}")
        confirm = input("确认删除？(y/n): ")
        if confirm.lower() == 'y':
            self.helper.execute("DELETE FROM student WHERE Id = %s", (sid,))
            print(f"✅ 已删除学号 {sid}")
        else:
            print("已取消")


class Course:
    """课程信息管理（对应 course_sys 表）"""

    def __init__(self, mysql_helper):
        self.helper = mysql_helper
        self.table = "course_sys"

    def course_menu(self):
        while True:
            print("-----------课程信息菜单----------")
            print("\n 1) 查看所有课程")
            print(" 2) 新增课程")
            print(" 3) 删除课程")
            print(" 4) 返回上一层菜单")
            no = input("请输入选择的序号：")

            if no == '1':
                self.show_courses()
                input("按任意键返回上层菜单")
            elif no == '2':
                self.add_course()
                input("按任意键返回上层菜单")
            elif no == '3':
                self.delete_course()
                input("按任意键返回上层菜单")
            elif no == '4':
                break
            else:
                print("输入错误！请重新输入！")

    def show_courses(self):
        sql = "SELECT * FROM course_sys"
        res = self.helper.find_all(sql)
        if not res:
            print("暂无课程数据")
            return
        print(f"\n{'='*50}")
        for c in res:
            print(f"  {c['cNo']}  {c['cName']}  (创建者: {c['createUser']})")
        print(f"{'='*50}")
        print(f"共 {len(res)} 门课程\n")

    def add_course(self):
        cno = input("课程编号: ")
        cname = input("课程名称: ")
        sql = "INSERT INTO course_sys (cNo, cName, createUser, createTime) VALUES (%s, %s, %s, NOW())"
        try:
            self.helper.execute(sql, (cno, cname, "admin"))
            print(f"✅ 新增课程成功: {cno} {cname}")
        except Exception as e:
            print(f"❌ 新增失败: {e}")

    def delete_course(self):
        cno = input("请输入要删除的课程编号: ")
        sql = "DELETE FROM course_sys WHERE cNo = %s"
        self.helper.execute(sql, (cno,))
        print(f"✅ 已删除课程 {cno}")


def main_menu():
    # 连接已存在的 db_0623_1 数据库
    my_sql = MySqlHelper(host="localhost", port=3306, user='root', pwd='8888', db_name='db_0623_1')
    if not my_sql.connect():
        print("数据库连接失败，程序退出")
        return

    print("---------欢迎使用学生管理系统v1.0--------------")
    print(f"数据库: {my_sql.db_name}")

    while True:
        print("\n 1) 学生信息")
        print(" 2) 课程信息")
        print(" 3) 退出\n")

        no = input("请输入选择的序号：")
        if no == '1':
            Student(my_sql).student_menu()
        elif no == '2':
            Course(my_sql).course_menu()
        elif no == '3':
            print("👋 再见！")
            break
        else:
            print("输入错误！请重新输入！")

    my_sql.close()


if __name__ == '__main__':
    main_menu()






---------欢迎使用学生管理系统v1.0--------------
数据库: db_0623_1

 1) 学生信息
 2) 课程信息
 3) 退出

-----------学生信息菜单----------

 1) 查询所有学生信息
 2) 新增学生信息
 3) 修改学生信息
 4) 删除学生信息
 5) 返回上一层菜单

ID     姓名         性别     出生       院系               地址                  
-----------------------------------------------------------------
901    张老大        男      2005     计算机系             北京市海淀区              
902    张老二        男      2006     中文系              北京市昌平区              
903    张三         女      2002     中文系              湖南省永州市              
904    李四         男      2003     英语系              辽宁省阜新市              
905    王五         女      2004     英语系              福建省厦门市              
906    王六         男      2002     计算机系             湖南省衡阳市              
907    66         女      2007     土木               cdu                 
908    888        难      2007     电电               cdu                 
909    999        男      2009     9                456                 
共 9 条记录

-----------学生信息菜单----------

 1) 查询所有